# 08 · Versionamento com Git e CI para API de ML

## Objetivos
- Versionar o projeto da API de ML com Git e GitHub.
- Praticar o fluxo mínimo: clone → branch → commit → push → Pull Request → merge → sincronização.
- Configurar **CI** (integração contínua) leve no GitHub Actions para dar feedback automático (lint e checagem de sintaxe) a cada push/PR.

## Escopo
- **Jenkins** e **CD** (entrega contínua) serão abordados **conceitualmente** ao final, sem execução prática nesta aula.

## Pré-requisitos do projeto
- Estrutura de uma **API de ML** já existente (ex.: diretório `api_idade/` com código da API, serviço de predição, artefatos de modelo, requirements, Dockerfile). Caso não exista, haverá nota sobre como reconstruir (sem executar aqui).
---

## 1. Git no ciclo de vida de ML: por que e quando usar

### Problema que Git resolve em projetos de ML
Projetos de ML evoluem simultaneamente em várias frentes: **código de API/serviço**, **scripts de treino/avaliação**, **dependências**, **configurações**, **artefatos de modelo** e, às vezes, **notebooks**. Sem controle de versão:
- Perde-se **histórico** (o que mudou entre versões).
- Perde-se **autoria** (quem mudou, quando e por quê).
- Perde-se o **racional técnico** (motivação e decisões documentadas).
- Rompe-se a **reprodutibilidade** (não há como reconstruir o resultado).

### O que é Git (definição e propriedades)
**Git** é um **sistema de controle de versão distribuído**. “Distribuído” significa que **cada clone** do repositório contém **todo o histórico** e metadados, sem depender de um servidor central para existir. Git registra **snapshots** do projeto chamados **commits** (estado dos arquivos + metadados como autor, data, mensagem e hash). A partir desses commits, é possível:
- **Comparar** versões (diferenciar linhas alteradas).
- **Reverter** mudanças (voltar a estados anteriores).
- **Ramificar** o trabalho (branches para evoluções paralelas).
- **Auditar** (rastrear autoria e motivação via mensagens/PRs).

### Benefícios específicos para ML/serving
- **Rastreabilidade**: vincular **versões de código e artefatos** aos resultados (métricas, releases) e a decisões tomadas.
- **Reprodutibilidade**: reconstruir ambiente e comportamento de **inferência** a partir de um **commit** (mesmos arquivos, mesmas versões).
- **Colaboração controlada**: **branches** isolam alterações; **Pull Requests (PRs)** formalizam revisão e registro de decisão.
- **Automação (CI)**: a cada *push/PR*, **pipelines** validam estilo, sintaxe e testes, detectando **quebras de contrato** (ex.: formato do payload de `/predict`) com antecedência.

### Quando usar Git no ciclo de vida de ML
- **Exploração inicial**: versionar scripts/notebooks que definem hipótese e pré-processamentos.
- **Treino/avaliação**: versionar pipelines de treino, configurações e **código determinístico** para repetir resultados.
- **Serviço/produção**: versionar a **API de inferência**, contratos de entrada/saída e **infra de empacotamento** (ex.: Dockerfile).
- **Operação/bugfix**: usar branches para correções sem bloquear evolução de features.

### O que versionar neste curso (decisão pragmática)
- **Código da API** (`main.py`, `services/`), **configuração leve**, **artefato mínimo** necessário para servir (pequeno, não sensível), `requirements.txt`, **Dockerfile**.
- **Documentação operacional** (ex.: `README.md` descrevendo endpoints, execução local e dependências).

### O que **não** versionar (e alternativas)
- **Dados volumosos** e **checkpoints grandes** → usar armazenamento externo (buckets/volumes) ou soluções específicas (ex.: DVC/Git LFS, registries de modelos).
- **Segredos** (tokens, chaves, `.env`) → usar **secret managers**/variáveis de ambiente.
- **Gerados temporários** (caches, `__pycache__`, saídas transitórias) → excluir via **`.gitignore`**.

### Analogia operacional
Pense no repositório como o **caderno de laboratório**: cada commit é uma **anotação datada** e verificável. A **CI** é o **checklist automatizado** que roda sempre que uma nova anotação é feita, garantindo que o experimento (serviço/ API) continua **executável e consistente**.

---

## 2. Conceitos essenciais de Git

### Elementos fundamentais
- **Repositório (repo)**: pasta com histórico versionado. Pode existir **localmente** (sua máquina) e **remotamente** (GitHub). O repositório local contém a **working tree** (arquivos em edição), o **.git** (metadados/histórico) e o **index** (staging).
- **Working tree**: seus arquivos “vivos” no disco. É onde edições ocorrem antes de versionar.
- **Index / Staging area**: área de preparação. Você **seleciona** quais mudanças irão compor o próximo **commit**.
- **Commit**: *snapshot* do projeto + metadados (autor, data, mensagem, **hash SHA** que identifica unicamente o commit).
- **Branch**: linha de desenvolvimento independente apontando para uma sequência de commits. A principal será **`main`**.
- **HEAD**: ponteiro local indicando “onde você está” (o commit/branch corrente).
- **Tag**: rótulo imutável para um commit (ex.: marcar releases).
- **Remoto `origin`**: referência nomeada para o repositório hospedado (GitHub).  
  - **`origin/main`**: **branch de rastreamento remoto** (estado de `main` no servidor).
- **Merge**: integração de uma branch em outra; pode ser *fast-forward* (sem commit extra) ou gerar um **commit de merge** (quando há divergência).
- **Fetch / Pull / Push**:
  - **Fetch**: busca atualizações do remoto **sem** alterar sua working tree.
  - **Pull** = **fetch + merge**: traz e integra na sua branch atual.
  - **Push**: envia seus commits locais para o remoto.

### Estados de arquivo (ciclo de vida)
- **Untracked** → arquivo novo, ainda não versionado.
- **Modified** → arquivo versionado com alterações locais pendentes.
- **Staged** → mudanças selecionadas para o próximo commit.
- **Committed** → mudanças registradas no histórico.

### Fluxo didático desta aula (mapeado a conceitos)
1) **Verificar Git e identidade** → garante metadados de autoria corretos nos **commits**.  
2) **Criar repositório no GitHub (remoto)** → estabelece o **`origin`**.  
3) **Inicializar repositório local** e padronizar **`main`** → define a **branch** principal e primeiro **commit**.  
4) **Conectar remoto e fazer push** → publica seu histórico local no **`origin`**.  
5) **Branch de feature** → cria uma **linha isolada** para uma alteração específica; depois abre **PR** para revisão.  
6) **Merge e sincronização** → integra a feature na **`main`** e alinha local com **`origin/main`** (pull).  
7) **CI no GitHub Actions** → em cada **push/PR**, o workflow executa validações automatizadas.

### Boas práticas para ML/serving
- **Uma branch por assunto** e PRs pequenos → facilita revisão e rollback.
- **Mensagens de commit** curtas e no **imperativo** (clareza e rastreabilidade).
- **Contrato de API** (entrada/saída) deve ser estável; alterações exigem **PR** detalhado e **validações** na CI.
- **.gitignore** assertivo para evitar vazamento de dados/segredos e ruído de arquivos temporários.

> Observação: **rebase**, **cherry-pick** e fluxos avançados existem, mas não são necessários para o escopo prático desta aula; manteremos foco em **branch → PR → merge** com **pull** para sincronizar.
---

## 3. Contexto do projeto usado na aula

Usaremos uma **API de ML** (ex.: `api_idade/`) com:
- `main.py` (ex.: FastAPI) e endpoints (`/health`, `/predict`).
- `services/` com a lógica de negócio (carrega pipeline/artefatos e executa `predict`).
- `artefatos/` com o **pipeline mínimo** necessário para inferência.
- `requirements.txt` e **Dockerfile** para empacotamento.

Se a pasta não existir, há uma **função geradora** que cria a estrutura a partir de um artefato já treinado (não executaremos nesta aula).

**Contrato de serviço** (essencial para CI e revisão): formato de entrada e saída do `/predict`. A cada mudança, PR + CI ajudam a evitar quebras de contrato (ex.: tipos, nomes de campos).

---

## 4. Fluxo de Git, plataformas e convenções de branch/commit

### 4.1 Git × GitHub/GitLab/Bitbucket (diferenças objetivas)
- **Git**: sistema de **controle de versão distribuído** que você usa **localmente** para versionar arquivos (linha de comando, histórico, branches, commits).
- **GitHub / GitLab / Bitbucket**: **plataformas de hospedagem** para repositórios Git (**remotos**) que adicionam colaboração (Pull/Merge Requests, Issues, permissões), automação (CI/CD), auditoria e interface web.
- **Analogia direta**:  
  - **Git** = o “**versor local**” que registra cada foto do projeto (commits).  
  - **GitHub/GitLab/Bitbucket** = a “**nuvem inteligente**” (tipo um Google Drive especializado) onde você **sincroniza** as fotos, compartilha com o time e automatiza checagens.

**Conclusão**: você trabalha no **repositório local** com Git; publica e colabora via **repositório remoto** numa dessas plataformas.

---

### 4.2 Fluxo prático fim a fim (visão geral antes dos detalhes)
**Cenário**: você já tem a pasta do projeto (ex.: a API de ML). Objetivo: colocar sob controle de versão e colaborar pelo GitHub.

1) **Criar repositório remoto** no GitHub (vazio; sem README/.gitignore automáticos).  
2) **Inicializar repositório local** na pasta do projeto, criar primeiro commit e padronizar a branch principal como **`main`**.  
3) **Conectar** o local ao **remoto** (adicionar `origin`) e fazer o **primeiro push**.  
4) **Criar uma branch de feature** para uma mudança específica (`feat/...`).  
5) **Editar arquivos** → **staging** (selecionar mudanças) → **commit** (registrar).  
6) **Push da branch** para o remoto e **abrir Pull Request (PR)** `feat/...` → `main`.  
7) **Revisar PR**, garantir **CI verde**, **mesclar** (*merge*) na `main`.  
8) **Sincronizar localmente** (voltar para `main` e `pull`) e **remover branch** de feature.

**Pontos-chave**:
- **Uma branch por assunto**.  
- **Commits pequenos** e mensagens claras.  
- **PRs pequenos** integram mais rápido e com menos risco.

---

### 4.3 Termos essenciais (depois do fluxo)
- **Repositório (repo)**: conjunto de arquivos + histórico de versões. Pode ser **local** (sua máquina) e **remoto** (GitHub).
- **Working tree**: seus arquivos “vivos” no disco (onde você edita).
- **Staging area (index)**: área de **preparação** do próximo commit (você escolhe o que entra, com `git add`).
- **Commit**: **snapshot** do projeto + metadados (autor, data, mensagem, hash). Unidade atômica do histórico.
- **Branch**: **linha de desenvolvimento** que aponta para uma sequência de commits. A principal será **`main`**. Permite trabalhar em paralelo sem quebrar a principal.
- **HEAD**: **ponteiro** que indica o commit/branch atual.
- **Remoto `origin`**: **apelido** para o repositório remoto (GitHub) ao qual você empurra/baixa mudanças.
- **Fetch / Pull / Push**:  
  - **Fetch**: baixa referências do remoto **sem** integrar na sua branch.  
  - **Pull** = fetch + merge: baixa **e integra** na branch atual.  
  - **Push**: envia seus commits locais para o remoto (publica suas mudanças).
- **Merge**: integra os commits de uma branch na outra (pode criar um commit de merge).
- **Pull Request (PR)**: proposta de integração de uma branch em outra no **remoto**, com descrição, revisão e checks (CI). “Porta de entrada” para a `main`.

---

### 4.4 Convenções de branches, commits e PRs (aplicáveis ao fluxo)
- **Branches por assunto** (prefixos padronizados):  
  - `feat/<descricao-curta>` → nova funcionalidade.  
  - `fix/<descricao-curta>` → correção de bug.  
  - `docs/<descricao-curta>` → documentação.  
  - `chore/<descricao-curta>` → tarefas operacionais (lint/CI/build).
  - Exemplos: `feat/validacao-payload-predict`, `fix/dtype-dataframe`, `docs/atualiza-readme`.
- **Commits**: mensagem curta, em **imperativo**, descrevendo **o que muda** (≤ 72 chars na 1ª linha).  
  - Exemplos: `docs: adiciona README mínimo`, `feat: valida payload no /predict`, `fix: corrige dtype no service`.
- **PRs**: objetivo, impacto, notas de teste, riscos. Só mesclar com **revisão** e **CI verde**.

---

## 5. Conta no GitHub, verificação/instalação do Git e configuração de identidade (Windows) — **sem conexão ao repositório** 

### 5.1 Criar conta no GitHub (pré-requisito para colaboração)
1) Acesse **https://github.com** → **Sign up**.  
2) Informe **e-mail**, defina **usuário** (curto/profissional) e **senha**.  
3) Ative **2FA** (aplicativo autenticador recomendado).  
4) Em **Settings → Emails**, mantenha visível (ou adicione) o e-mail que você usará nos **commits** (precisa corresponder ao `user.email` do Git para os commits aparecerem no seu perfil).  

> Nesta etapa **não** conecte repositórios; apenas garanta que sua conta existe e está configurada.

---

### 5.2 Verificar se o Git está instalado e acessível no PATH
- **Objetivo**: confirmar se o executável `git` está disponível no terminal.
- **Comandos**:
  - `git --version` → deve retornar algo como `git version X.Y.Z`.  
    - Se retornar “comando não encontrado”, o Git não está instalado ou não está no **PATH**.
  - (Diagnóstico Windows) `where git` ou `Get-Command git` (PowerShell) → mostra o caminho do executável.
- **Ação**: se **não** estiver instalado, prossiga para 5.3.

---

### 5.3 Instalar o Git (Windows)
- **Winget (Windows 10/11)**  
  - `winget install --id Git.Git -e --source winget`  
  - Instala **Git for Windows**, incluindo **Git Bash** e **Git Credential Manager**.
- **Chocolatey** (se já existir)  
  - `choco install git -y`
- **Instalador oficial (GUI)**  
  - Baixe o **Git for Windows** (site oficial), execute o instalador e mantenha:  
    - **Adicionar Git ao PATH**,  
    - **Git Credential Manager** (útil para autenticação futura na etapa 6),  
    - **Git Bash**.

> Após instalar, **feche e reabra** o terminal e repita `git --version`. Se falhar, revise o **PATH**.

---

### 5.4 Configurar **identidade global** (metadados de autoria)
- **Objetivo**: cada **commit** registra autor e e-mail. Use o **mesmo e-mail** configurado no GitHub (5.1) para que seus commits apareçam no perfil.
- **O que definir (escopo global do usuário)**:
  - `user.name` → seu nome legível (ex.: *Maria Silva*).  
  - `user.email` → e-mail associado à conta GitHub (ex.: *maria@exemplo.com*).  
- **Por que global?** Vale para todos os repositórios do seu usuário na máquina. Pode ser sobrescrito **por repositório** se necessário.

---

### 5.5 Padrões recomendados (globais)
- **Branch padrão de novos repositórios**  
  - `init.defaultBranch main` → garante que repositórios recém-inicializados usem **`main`** (evita `master`).
- **Quebra de linha no Windows**  
  - `core.autocrlf true` → converte `CRLF`↔`LF` automaticamente para evitar *diffs* falsos ao colaborar com Unix.
- **Inspeção das configurações**  
  - `git config --global --list` → confirme `user.name`, `user.email`, `init.defaultBranch`, `core.autocrlf`.

> Observação: **credential helper** (armazenador de credenciais) é útil, mas a autenticação prática (PAT/SSH) e o **vínculo ao repositório remoto** serão tratados **na etapa 6**.

---

### 5.6 Erros comuns e correções
- **`git: command not found`** → instalar Git ou ajustar **PATH** (reabra o terminal após instalar).  
- **Commits não aparecem no perfil GitHub** → `user.email` diferente do e-mail visível no GitHub; ajuste e use nas próximas alterações.  
- **Branch inicial `master`** ao criar repositório local → defina `init.defaultBranch main` e reinicialize, ou renomeie com `git branch -m master main`.  
- **Conflitos de fim de linha** em equipe mista Windows/Unix → padronize `core.autocrlf` (ou `.gitattributes`) conforme política do time.

In [ ]:
# Bloco de código 5 — Verificação, instalação e configuração de identidade (Windows)
# Executar no PowerShell ou Windows Terminal. Ajuste valores em MAIÚSCULAS quando houver.

# 1) Verificar instalação e PATH
git --version               # Deve imprimir 'git version X.Y.Z'
where git                   # Localiza o executável (cmd/PowerShell)
Get-Command git             # Alternativa no PowerShell, mostra o caminho

# 2) (Opcional) Instalar Git — execute UMA das opções abaixo se 'git --version' falhar
# 2.a) Winget (Windows 10/11)
# winget install --id Git.Git -e --source winget

# 2.b) Instalador oficial (GUI)
# - Baixe 'Git for Windows' no site oficial e aceite:
#   * Adicionar ao PATH
#   * Git Credential Manager
#   * Git Bash
# - Feche e reabra o terminal, então:
# git --version

# 3) Configurar identidade global (autoria dos commits)
git config --global user.name  "SEU_NOME_COMPLETO"     # Ex.: "Maria Silva"
git config --global user.email "SEU_EMAIL@EXEMPLO.COM" # Mesmo e-mail visível no GitHub

# 4) Definir padrões globais recomendados
git config --global init.defaultBranch main  # Usa 'main' como branch inicial em novos repositórios
git config --global core.autocrlf true       # Normaliza quebras de linha em ambiente Windows

# 5) Conferir configurações aplicadas
git config --global --list   # Verifique user.name, user.email, init.defaultBranch, core.autocrlf

## 6. Inicialização do repositório local

### 6.1 Onde rodar
- Rodar **na raiz do projeto** (pasta que contém os notebooks e a pasta `api_idade/`).  
- **Não** rodar dentro de `api_idade/`.

Estrutura esperada (forma hierárquica, sem caracteres especiais):
- raiz do projeto  
  - 06_treinar_modelo_motar_api.ipynb  
  - 07_docker_containerizacao_da_api.ipynb  
  - 08_git_e_ci_para_api_ml.ipynb  
  - api_idade/  
    - main.py  
    - services/  
    - preprocessing/  
    - utils/  
    - models/  
    - artefatos/  
    - Dockerfile  
    - .dockerignore  
  - dados/  
  - artefatos/  
  - processes_top.csv  

### 6.2 Objetivo
- Tornar a **raiz** um repositório Git, padronizar a branch inicial como **`main`**.  
- Usar um **`.gitignore` “restrictivo”**: ignorar quase tudo e **liberar apenas** `api_idade/` (mais `.gitignore` e, opcionalmente, `README.md`).

### 6.3 Estratégia de versionamento mínimo
- `.gitignore` na **raiz** com as regras:
  - Ignorar **tudo** (`*`).
  - **Permitir** (`!`) apenas:
    - o próprio `.gitignore` (para ser versionado);
    - `api_idade/` (e todo o seu conteúdo);
    - `README.md` (opcional).
- Dentro de `api_idade/`, você **pode** manter outro `.gitignore` específico do subprojeto, se desejar granularidade.

### 6.4 `.gitignore`: conceito e riscos
- É um arquivo de **padrões de exclusão** que diz ao Git o que **não** deve ser rastreado.
- **Útil**: evita versão de caches, temporários, *checkpoints*, segredos e dados volumosos.
- **Riscos**:
  - **Não apaga histórico**: se já versionou um segredo, o `.gitignore` **não remove**; é preciso reescrever histórico.
  - **Não retira arquivos já rastreados**: após adicionar regra, use `git rm --cached` para parar de rastrear o que já entrou.
  - **“Ignorar tudo” exige whitelists**: qualquer novo item na raiz **não será visto** até você liberar explicitamente no `.gitignore`. Planeje revisões do `.gitignore` quando adicionar `.github/`, documentação, etc.


In [5]:
# 6.A — Criar .gitignore na RAIZ com Python (executar a partir do notebook na RAIZ)
from pathlib import Path

raiz = Path.cwd()

conteudo = r"""
# Ignorar tudo na raiz por padrão
*

# Permitir o próprio .gitignore
!.gitignore

# (Opcional) permitir README na raiz
!README.md

# Permitir a pasta da API (e todo o seu conteúdo)
!api_idade/
!api_idade/**

# Dentro de api_idade/, ainda ignorar caches/temporários comuns (regrinhas locais)
api_idade/__pycache__/
api_idade/**/*.pyc
api_idade/.ipynb_checkpoints/
api_idade/.mypy_cache/
api_idade/.pytest_cache/
api_idade/.ruff_cache/
api_idade/logs/
api_idade/*.log

# Não versionar segredos/variáveis
api_idade/.env
api_idade/.env.*

# Dados/artefatos grandes DENTRO da API (ajuste se necessário)
api_idade/data/
api_idade/datasets/
api_idade/models/
api_idade/outputs/
api_idade/artifacts/

# Exceção: manter o artefato mínimo necessário para servir a API (ajuste o caminho exato)
!api_idade/artefatos/
!api_idade/artefatos/pipeline_referencia.pkl

# IDEs/Edits
.vscode/
.idea/
*.code-workspace
"""

(raiz / ".gitignore").write_text(conteudo.strip() + "\n", encoding="utf-8")
print(f".gitignore criado/atualizado em: {(raiz / '.gitignore').resolve()}")


.gitignore criado/atualizado em: D:\Downloads\aula 4\.gitignore



### 6.5 Passo a passo (comandos e funções)
1) Confiar que as pastas são seguras.  
   - git config --global --add safe.directory "D:/Downloads/aula 4"
   - git config --global --add safe.directory "D:/Downloads/aula 4/api_idade"

2) Inicializar o repositório.  
   - `git init`: cria a pasta oculta `.git/` e prepara o controle de versão **na raiz**.

3) Padronizar a branch inicial.  
   - `git branch -m main`: renomeia a branch corrente para **`main`** .

4) Criar o `.gitignore` **na raiz** (rodar a célula Python de cima).  
   - Objetivo: aplicar a política “ignorar tudo, liberar `api_idade/`”.

5) Conferir o que o Git enxerga.  
   - `git status`: lista arquivos não rastreados (*untracked*) e mudanças.

6) Adicionar ao *staging*.  
   - `git add .gitignore api_idade`: seleciona `.gitignore` e a pasta `api_idade/` para o próximo commit.

7) Criar o primeiro commit.  
   - `git commit -m "init: estrutura da API pronta para versionamento"`: grava um **snapshot** com mensagem curta.

8) Ver histórico (opcional).  
   - `git log --oneline -n 3`: lista os últimos commits de forma resumida.

### 6.6 Observações operacionais
- Pastas como `dados/`, `artefatos/` (na raiz) e arquivos como `processes_top.csv` **não** serão versionados com a política “ignorar tudo”, a menos que você os libere depois.  
- Quando evoluir para CI, será necessário **permitir** `.github/workflows/` no `.gitignore`.


In [ ]:
# 6.B — Comandos de terminal 

# 0) Confiar no caminho com espaço (CMD exige aspas duplas)
git config --global --add safe.directory "D:/Downloads/aula 4"
git config --global --add safe.directory "D:/Downloads/aula 4/api_idade"

# 1) Inicializar o repositório Git na RAIZ
git init       # cria .git/ e começa a rastrear mudanças neste diretório

# 2) Padronizar a branch inicial como 'main'
git branch -m main   # renomeia a branch corrente para 'main' (seguro se já for 'main')

# 3) Ver o estado (o que está visível para o Git após aplicar o .gitignore)
git status

# 4) Adicionar ao staging (seleciona o que vai para o próximo commit)
git add .gitignore api_idade # Se quiser sempre adicionar tudo, pode usar "git add ."

# 6) Criar o primeiro commit (snapshot com mensagem objetiva)
git commit -m "init: estrutura da API pronta para versionamento"

# 7) Verificar o histórico (opcional)
git log --oneline -n 3


## 7. Repositório remoto no GitHub — **HTTPS + PAT** (único método adotado)

### 7.1 Criar o repositório no GitHub (UI)
1) Acesse `https://github.com` e autentique-se.  
2) Clique no ícone **+** (canto superior direito) → **New repository**.  
3) **Owner**: sua conta. **Repository name**: `api_idade` (ou o nome desejado).  
4) **Visibility**: *Public* (demo) ou *Private* (turma).  
5) **Desmarque**: “Add a README file”, “Add .gitignore”, “Choose a license” (evita conflito com o primeiro push local).  
6) Clique **Create repository**. Na tela **Quick setup**, mantenha aberta a **URL HTTPS** (será usada para conectar do local ao remoto).

---

## 7.2 Gerar o PAT (Personal Access Token) — guia preciso (HTTPS + PAT)

**Finalidade**: permitir `push/pull` por **HTTPS** sem senha de conta, usando um **token** salvo com segurança pelo **Git Credential Manager** (GCM) no Windows.

**Passo a passo (Fine-grained recomendado)**  
1) GitHub (logado) → **Settings** → **Developer settings** → **Personal access tokens** → **Fine-grained tokens** → **Generate new token**.  
2) **Token name**: um identificador claro (ex.: `api_idade_push`).  
3) **Expiration**: escolha curta (ex.: 7–30 dias) para aula/demonstração.  
4) **Resource owner**: sua conta.  
5) **Repository access**: **Only select repositories** → selecione o repositório criado (`api_idade`).  
6) **Repository permissions** (mínimo para push):  
   - **Contents**: **Read and write**.  
   - **Metadata**: **Read-only** (geralmente padrão).  
   - *(Opcional)* **Pull requests**: **Read and write** (se pretende interagir com PRs via API).  
7) Clique **Generate token** → **copie** o token exibido **(aparece uma única vez)**.  
8) **Guarde** o token com segurança. **Não** versione, **não** compartilhe.

> Alternativa se Fine-grained não estiver disponível/permitido: **Tokens (classic)** → **Generate new token (classic)** → marque o escopo **repo** → defina expiração → gere e **copie**.


---

## 7.3 Conectar local → remoto (HTTPS + PAT) — guia operacional

**Pré-requisitos**  
- Repositório local **inicializado** e com **primeiro commit**.  
- URL **HTTPS** do repositório (`https://github.com/SEU_USUARIO/api_idade.git`).  
- **PAT** gerado (Seção 7.2).

**Passos**  
1) *(Somente se houver erro por caminho com espaço)* Autorize o diretório no Git (safe.directory):  
   `git config --global --add safe.directory "D:/Downloads/aula 4"`  
2) Adicione/ajuste o **remoto** `origin` com a **URL HTTPS**.  
3) Execute `git push -u origin main`. No prompt:  
   - **Username**: seu usuário GitHub.  
   - **Password**: **cole o PAT**. O **GCM** salvará o token (próximos pushes/pulls não pedirão).  
4) Valide no GitHub: abas **Code/Commits/Branches**.  
5) Confirme **Default branch = `main`** em **Settings → Branches**.  
6) *(Opcional)* Proteja `main` (branch protection) para exigir PR + CI verde antes de merge.


In [ ]:
#  Bloco 7 — Conectar repositório local ao remoto (HTTPS + PAT) — Windows Terminal/CMD
#  Pré: repositório local inicializado e com 1º commit (etapa 6); PAT já gerado.

#  0) Ir para a RAIZ do projeto
cd /d "D:\Downloads\aula 4"

#  1) (Opcional) Autorizar caminho com espaço se houver alerta 'safe.directory'
#  git config --global --add safe.directory "D:/Downloads/aula 4"

#  2) Definir o remoto 'origin' com a URL HTTPS do repositório (ajuste USUÁRIO/REPO)
git remote remove origin 2>nul
git remote add origin https://github.com/SEU_USUARIO/api_idade.git

#  3) Conferir URLs configuradas para fetch/push
git remote -v

#  4) Publicar 'main' e definir upstream (-u)
#  Na primeira vez, o Git Credential Manager exibirá a autenticação:
#  Username = seu usuário GitHub | Password = cole o PAT
git push -u origin main


## 8. Branch de feature, alteração simples e Pull Request

### O que será feito (ordem e propósito)
1) Garantir que você está na **raiz do projeto** e na **branch `main`** atualizada.  
2) Criar uma **branch de trabalho** (`dev`, para o exemplo) a partir do estado atual da `main`.  
3) Criar/editar um arquivo simples (`README.md`) para representar a mudança mínima.  
4) **Adicionar** a alteração ao *staging* e **criar um commit** com mensagem curta e objetiva.  
5) **Publicar** a branch no remoto com `-u` (define o *upstream*, facilitando futuros `push/pull`).  
6) Abrir um **Pull Request** no GitHub de `dev` → `main`, descrever objetivo/impacto, aguardar revisão/CI e **fazer o merge**.

### Comandos e explicações (executar na RAIZ do projeto)
- `git checkout main`  
  Troca para a branch principal local.
- `git pull --ff-only`  
  Sincroniza a `main` com o remoto (só *fast-forward*; evita criar commit de merge local inesperado).
- `git checkout -b dev`  
  Cria **e troca** para a nova branch `dev` (filha do commit atual da `main`).
- `git add README.md`  
  Seleciona a alteração do `README.md` para entrar no próximo commit (staging).
- `git commit -m "docs: adiciona README"`  
  Cria o commit com mensagem curta, no imperativo, descrevendo **o que** mudou.
- `git push -u origin dev`  
  Publica a branch `dev` no repositório remoto e define o *upstream* (`origin/dev`), permitindo futuros `git push`/`git pull` sem parâmetros adicionais.

**Após publicar**: abrir o PR no GitHub (branch `dev` → `main`), revisar/mesclar e seguir para a sincronização (Seção 9).


In [6]:
# (1/2) Criar/atualizar README.md na RAIZ do projeto
from pathlib import Path

conteudo = """# API Idade

## Endpoints
- GET /health
- POST /predict

## Execução local (exemplo)
1) Criar e ativar um ambiente virtual.
2) Instalar dependências de api_idade/requirements.txt.
3) Subir a API:
   uvicorn api_idade.main:app --host 0.0.0.0 --port 8010 --reload
"""

Path("README.md").write_text(conteudo, encoding="utf-8")
print("README.md criado/atualizado na raiz.")


README.md criado/atualizado na raiz.


In [ ]:
# Bloco 8.2 — Branch de trabalho, alteração mínima e publicação — Windows Terminal/CMD
# Objetivo: criar 'dev', commitar README e subir para PR.

# 0) Ir para a RAIZ do projeto
cd /d "D:\Downloads\aula 4"

# 1) Garantir que 'main' local está atualizada
git checkout main
git pull --ff-only

# 2) Criar e trocar para a branch de trabalho
git checkout -b dev

# 3) Selecionar e commitar alteração mínima (README)
git add README.md
git commit -m "docs: adiciona README mínimo"

# 4) Publicar 'dev' no remoto e definir upstream (-u)
git push -u origin dev

# 5) Abrir PR 'dev' → 'main' no GitHub (UI), revisar e mesclar

## 9. Sincronização local após o merge

### O que é e por que fazer
- **Merge no PR**: o GitHub integra os commits de `dev` em `main` no **remoto**.
- **Desalinhamento local**: sua `main` local pode estar atrás de `origin/main`. É necessário **atualizar**.

### Comandos (e função de cada um)
1) `git fetch --prune origin` → atualiza refs do remoto e remove rastros de branches apagadas.  
2) `git checkout main` → volta para a branch principal **local**.  
3) `git pull --ff-only` → avança a `main` local até `origin/main` **sem** criar commit de merge local.  
4) `git branch -d dev` → remove a branch **local** já integrada (seguro: recusa se não estiver mesclada).  
5) `git push origin --delete dev` → apaga a branch **remota** (se o PR não apagou).  
6) `git remote prune origin` → limpeza final de referências remotas obsoletas.


In [ ]:
# Bloco 9 — Sincronização local após o merge — Windows Terminal/CMD
# Objetivo: alinhar o repositório local depois que o PR foi mesclado em 'main' no GitHub.

# 0) Ir para a RAIZ do projeto
cd /d "D:\Downloads\aula 4"

# 1) Atualizar referências do remoto e limpar rastros de branches removidas
git fetch --prune origin

# 2) Voltar para 'main' local e avançar até 'origin/main' (somente fast-forward)
git checkout main
git pull --ff-only

# 3) Remover branch local já integrada (seguro: '-d' recusa se não estiver mesclada)
git branch -d dev

# 4) (Se a branch remota não tiver sido apagada pela UI do PR) remover no remoto
git push origin --delete dev

# 5) Limpeza final de referências remotas obsoletas
git remote prune origin


## 10. Integração Contínua (CI) — o “checklist automático” do projeto

**Definição prática.** Integração Contínua (CI) é uma **esteira automática de verificações** que roda **toda vez que alguém envia código** para o repositório (ao fazer *push* ou abrir um Pull Request). A ideia é responder rapidamente: “**o projeto continua íntegro?**”.

**Analogia.** Pense em uma **cozinha profissional**. Antes de abrir o salão, há um **checklist**: limpeza, equipamentos ligados, insumos conferidos. A **CI** é esse checklist aplicado ao **código**: não abre o “salão” (integração do código) se algo básico estiver errado.

### O que a turma já faz e o que CI cobre
Vocês já possuem **validações automáticas de dados e modelos** (ex.: *data drift*, aprovações e rastreio com **MLflow**). Isso responde:  
- “**Os dados estão dentro do contrato?**” (esquema, faixas, tipos)  
- “**O modelo está aprovado?**” (métricas, comparações, versão, artefatos)

A **CI**, neste curso, foca principalmente no **serviço de inferência** (a **API**) e no **empacotamento** básico:
- **Código compila?** (checagem de sintaxe)
- **Estilo/erros triviais?** (analisadores estáticos)
- **Contratos de API preservados?** (ex.: formato de entrada/saída do `/predict`)
- **Dependências declaradas e instaláveis?** (conferência de `requirements.txt`)
- **Empacotamento mínimo coerente?** (ex.: *Dockerfile* e layout de pastas)

> **Resumo:** as **regras de “subir ou não” o modelo/dados** já vêm do seu fluxo de *drift* e **MLflow**. A **CI** entra para garantir que **a camada de serviço** que entrega o modelo **não quebre** por erro básico de código, dependência ou contrato de API.

### Como a CI costuma se organizar (sem jargão)
- **Gatilho**: o que dispara a esteira (ex.: *push* na branch principal ou PR aberto).
- **Máquina executora**: onde as verificações rodam (fornecedor gerenciado ou servidor próprio).
- **Passos do checklist** (sequência simples e rápida):
  1) **Baixar o código** do repositório.
  2) **Configurar o ambiente** (ex.: versão do Python).
  3) **Instalar ferramentas leves** (ex.: linter/analisador).
  4) **Rodar verificações baratas** (estilo/sintaxe).
  5) **Rodar testes objetivos** da API (ex.: validar esquema de entrada/saída).
  6) **(Opcional)** montar o pacote/contêiner **sem publicar** (apenas para detectar erros cedo).

### Boas práticas para nossa realidade
- **Curto e direto**: a esteira deve durar **poucos minutos**.
- **Falhar cedo**: sintaxe/estilo antes de qualquer coisa mais lenta.
- **Evitar treinos pesados**: **não** treinar modelos na CI da API.
- **Mensagens claras** no relatório: o que falhou e como corrigir.
- **Usar o que já existe**: se MLflow já guarda versões/artefatos, CI só **garante que a API consome** corretamente esses artefatos aprovados.

### Indicadores simples de qualidade da CI
- **Tempo médio** da esteira.
- **Taxa de falha por erro simples** (bom sinal, evita levar erro bobo para PR).
- **Número de PRs bloqueados por quebra de contrato** (ajuda a manter padrões da API).


## 11. Entrega Contínua (CD) — preparar e promover versões com segurança

**Definição prática.** Entrega Contínua (CD) é a **automação que empacota uma versão pronta** (por exemplo, uma imagem da API com o modelo embarcado) e a **prepara para ir ao ar**, com **pontos de controle** e possibilidade de **voltar atrás** se algo der errado.

**Analogia.** Imagine uma **linha de envase**. A garrafa (sua versão da API) é enchida, lacrada, rotulada e **vai para a conferência final** antes de chegar às prateleiras. A **CD** cuida desse **envase + checagens + liberação**.

### Continuous Delivery × Continuous Deployment
- **Delivery** (*entrega contínua*): a esteira **deixa tudo pronto**; alguém **aprova manualmente** a ida para produção.
- **Deployment** (*implantação contínua*): a promoção para produção acontece **automaticamente**, se passar pelos critérios.

> Para nossa realidade de aula, pense em **Continuous Delivery**: a pipeline deixa a versão pronta; a **liberação** é **decisão humana** (professor/equipe).

### O que a turma já cobre e o que a CD acrescenta
Vocês já **aprovam dados e modelos** com *data drift* e **MLflow** (métrica, versão, artefatos). A **CD** **não reavalia modelo/dado**; ela:  
- **Empacota** a API + artefatos aprovados (ex.: **construir a imagem Docker** com código e modelo versionados).  
- **Rotula** a versão (ex.: tag com número/commit).  
- **Prepara a ida** a um ambiente (ex.: *staging*, depois produção).  
- **Executa verificações rápidas** após subir (ex.: *health check* da API).  
- **Prevê retorno** rápido (rollback) se houver problema.

### Fluxo prático (API de inferência)
1) **Build do artefato implantável**: imagem Docker da API usando o **modelo aprovado no MLflow**.  
2) **Identidade da versão**: tag clara (ex.: `api-idade:1.3.0` ou `api-idade:commit-sha`).  
3) **Publicação interna**: enviar a imagem ao repositório de imagens (registry).  
4) **Promoção controlada**: primeiro ambiente de teste (ex.: *staging*), depois produção.  
5) **Verificações pós-subida**: endpoint `/health`, chamada de `/predict` com amostra conhecida, latência.  
6) **Estratégia de liberação** (quando for avançar):  
   - **Blue–green**: ambiente B novo; troca o tráfego de A→B quando está saudável.  
   - **Canário**: libera para **uma pequena parcela** dos usuários, expande gradualmente.  
7) **Rollback**: se as métricas degringolarem, **volta** para a versão anterior.

### Boas práticas com modelos
- **Travar a versão do artefato**: a mesma versão que passou no MLflow é **a que vai** na imagem.  
- **Contrato estável** do `/predict`**:** se mudar, documentar e validar com checagens simples (ex.: schema).  
- **Observabilidade**: logs de erro e latência, amostras sentinela, alarmes simples.

> **Resumo:** **CD** não decide se o **modelo** é bom (isso vocês já fazem); ela **padroniza o empacotamento e a promoção** da **API** que serve esse modelo, com botões de **aprovar** e **voltar**.


## 12. Jenkins — quando faz sentido e por que não usaremos agora

**Definição prática.** **Jenkins** é um **servidor de automação** que você **instala e administra** para orquestrar **esteiras de CI/CD**. Ele executa **pipelines** descritas em arquivo (geralmente `Jenkinsfile`) em **máquinas executoras** controladas por você.

**Analogia.** Pense em um **gerente de fábrica próprio**: você decide **onde fica a fábrica**, **quem são os operadores**, **quais máquinas** usar e **como** a linha deve funcionar. Isso dá **muito controle**, mas também traz **custo de cuidar** de tudo (atualizações, permissões, segurança, backups).

### Componentes em termos simples
- **Controlador (servidor Jenkins)**: agenda e coordena tarefas, guarda histórico e configurações.  
- **Agentes (máquinas executoras)**: onde as etapas rodam (podem ser servidores locais, VMs em nuvem, contêineres, Kubernetes).  
- **Pipeline**: o roteiro da esteira (em `Jenkinsfile`, com estágios como “baixar código”, “instalar ferramentas”, “testar”, “empacotar”, “publicar”).

### Por que equipes escolhem Jenkins
- **Ambiente restrito** ou **sem internet** (ex.: redes corporativas fechadas): você precisa rodar tudo “em casa”.  
- **Integrações legadas** ou específicas: necessidade de plugins e controle fino.  
- **Escala customizada**: muitos agentes, filas, paralelismo, mistura de sistemas operacionais.

### Desvantagens para nosso cenário de aula
- **Instalação e manutenção**: demanda tempo, permissões e monitoramento.  
- **Curva de configuração**: segurança (credenciais/segredos), isolamento, *hardening*.  
- **Overhead desnecessário** quando um serviço gerenciado (ex.: CI integrado ao Git) já atende.

### Jenkins aplicado a ML/serving (visão conceitual)
- **CI**: baixar código, validar API (sintaxe/estilo/contratos), testar.  
- **Empacotar**: construir imagem Docker com **modelo aprovado** no MLflow.  
- **CD**: publicar imagem no registry, promover para *staging*/produção com verificações e possibilidade de rollback.  
- **Integração com MLflow**: baixar **artefatos aprovados** por tag/versão para montar a imagem.

> **Por que não usaremos agora.** Laboratório sem instalação, tempo limitado, e **não é o foco** desta aula. O objetivo aqui é **entender os papéis**: vocês **já cuidam dos dados/modelos** (drift + MLflow); faltava mostrar **onde a API entra** (CI de serviço e, futuramente, CD). Quando houver ambiente adequado, o Jenkins (ou outra ferramenta) pode **orquestrar** isso de ponta a ponta.
